In [4]:


import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path('..')
PROCESSED = BASE / 'data' / 'processed'

fund_master = pd.read_csv(PROCESSED / 'fund_master_ingested.csv')
nav_history = pd.read_csv(PROCESSED / 'nav_history_ingested.csv')

fund_master.columns = (
    fund_master.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
)

nav_history.columns = (
    nav_history.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
)

print(fund_master.columns.tolist())
print(nav_history.columns.tolist())

fund_master['launch_date'] = pd.to_datetime(
    fund_master['launch_date'],
    errors='coerce'
)

nav_history['date'] = pd.to_datetime(
    nav_history['date'],
    errors='coerce'
)

print(fund_master['launch_date'].dtype)
print(nav_history['date'].dtype)

print('Fund master shape:', fund_master.shape)
print('NAV history shape:', nav_history.shape)

print()
print('Fund master missing values')
print(fund_master.isnull().sum())

print()
print('NAV history missing values')
print(nav_history.isnull().sum())

fund_master_before = len(fund_master)
nav_before = len(nav_history)

fund_master = fund_master.drop_duplicates()
nav_history = nav_history.drop_duplicates()

print('Fund master duplicates removed:',
      fund_master_before - len(fund_master))

print('NAV history duplicates removed:',
      nav_before - len(nav_history))

fund_master = fund_master.dropna(subset=['amfi_code'])
nav_history = nav_history.dropna(subset=['amfi_code'])

fund_master['amfi_code'] = (
    fund_master['amfi_code']
    .astype(int)
)

nav_history['amfi_code'] = (
    nav_history['amfi_code']
    .astype(int)
)

nav_history['nav'] = pd.to_numeric(
    nav_history['nav'],
    errors='coerce'
)

nav_history = nav_history.dropna(subset=['nav'])
nav_history = nav_history[
    nav_history['nav'] > 0
]

print(nav_history['nav'].describe())

nav_history = nav_history.sort_values(
    ['amfi_code', 'date']
)



nav_history['nav'] = (
    nav_history.groupby('amfi_code')['nav']
    .ffill()
    .bfill()
)

cleaned_nav = []

for code, df in nav_history.groupby('amfi_code'):

    full_dates = pd.date_range(
        start=df['date'].min(),
        end=df['date'].max(),
        freq='B'
    )

    df = (
        df.set_index('date')
        .reindex(full_dates)
        .rename_axis('date')
        .reset_index()
    )

    df['amfi_code'] = code
    df['nav'] = df['nav'].ffill().bfill()

    cleaned_nav.append(df)

nav_history = pd.concat(
    cleaned_nav,
    ignore_index=True
)

print(nav_history.head())

q1 = nav_history['nav'].quantile(0.25)
q3 = nav_history['nav'].quantile(0.75)

iqr = q3 - q1

lower = q1 - 3 * iqr
upper = q3 + 3 * iqr

before = len(nav_history)

nav_history = nav_history[
    (nav_history['nav'] >= lower) &
    (nav_history['nav'] <= upper)
]

print('Outliers removed:',
      before - len(nav_history))

categorical_columns = [
    'fund_house',
    'scheme_name',
    'category',
    'sub_category',
    'plan',
    'benchmark',
    'fund_manager',
    'risk_category'
]

for col in categorical_columns:
    if col in fund_master.columns:
        fund_master[col] = (
            fund_master[col]
            .astype(str)
            .str.strip()
        )

if 'expense_ratio_pct' in fund_master.columns:

    fund_master['expense_ratio_pct'] = pd.to_numeric(
        fund_master['expense_ratio_pct'],
        errors='coerce'
    )

    fund_master['expense_ratio_pct'] = (
        fund_master
        .groupby('category')['expense_ratio_pct']
        .transform(
            lambda x: x.fillna(x.median())
        )
    )

print(
    fund_master['expense_ratio_pct']
    .isnull()
    .sum()
)

print('Fund master shape:',
      fund_master.shape)

print('NAV history shape:',
      nav_history.shape)

print()

print('Fund master missing values')
print(fund_master.isnull().sum())

print()

print('NAV history missing values')
print(nav_history.isnull().sum())

print()

duplicates = nav_history.duplicated(
    subset=['amfi_code', 'date']
).sum()

print('Duplicate NAV records:',
      duplicates)

nav_history = nav_history.sort_values(
    ['amfi_code', 'date']
)

nav_history['daily_return'] = (
    nav_history
    .groupby('amfi_code')['nav']
    .pct_change()
)

print(nav_history.head())


### Save cleaned datasets


fund_master.to_csv(
    PROCESSED / 'fund_master_clean.csv',
    index=False
)

nav_history.to_csv(
    PROCESSED / 'nav_history_clean.csv',
    index=False
)

print('=' * 60)
print('Data cleaning completed successfully')
print('=' * 60)

print('Clean fund records:',
      len(fund_master))

print('Clean NAV records:',
      len(nav_history))

print('Unique funds:',
      nav_history['amfi_code'].nunique())

print('Date range:',
      nav_history['date'].min().date(),
      'to',
      nav_history['date'].max().date())

print()

print('Files saved:')

print(PROCESSED / 'fund_master_clean.csv')
print(PROCESSED / 'nav_history_clean.csv')

print('=' * 60)
## Cleaning actions performed




['amfi_code', 'fund_house', 'scheme_name', 'category', 'sub_category', 'plan', 'launch_date', 'benchmark', 'expense_ratio_pct', 'exit_load_pct', 'min_sip_amount', 'min_lumpsum_amount', 'fund_manager', 'risk_category', 'sebi_category_code']
['amfi_code', 'date', 'nav']
datetime64[ns]
datetime64[ns]
Fund master shape: (40, 15)
NAV history shape: (46000, 3)

Fund master missing values
amfi_code             0
fund_house            0
scheme_name           0
category              0
sub_category          0
plan                  0
launch_date           0
benchmark             0
expense_ratio_pct     0
exit_load_pct         0
min_sip_amount        0
min_lumpsum_amount    0
fund_manager          0
risk_category         0
sebi_category_code    0
dtype: int64

NAV history missing values
amfi_code    0
date         0
nav          0
dtype: int64
Fund master duplicates removed: 0
NAV history duplicates removed: 0
count    46000.000000
mean       269.570265
std        577.187060
min         26.136600
